In [1]:
# PHASE 2 - ABLATION 8c: PROBE BATTERY on the frozen features (Part-I "Study A" probes, patch-level regime): Phikon ViT-B
# What the features encode: (2) HOSPITAL from the window-mean feature [logistic regression and MLP], (3) HOSPITAL from raw colour statistics (no model, shared baseline),
# (4) grid POSITION (0-8) of a patch from its own feature, (5) PATIENT identity (closed set).  Chance levels are printed.
# Models: ORIGINAL | SimCLR control | self | ADJACENCY | CCSM ms / ms+Fourier (ResNet18 also ms-random) (+ Lunit combined); seeds 42/43/44.   Kaggle: GPU on, Internet on.
# Checkpoints are found/downloaded automatically; for the combined Lunit arm attach the OUTPUT of phase2_ABL5_lunit_combined_adjacency_ccsm (skipped if absent).
# Estimated runtime ~5-6 h (16 models x ~20 min).  Resumable (a small .json per model); stops cleanly before 11 h -> Save Version and re-run with that OUTPUT attached as an input.
!pip install -q timm datasets kagglehub transformers


import os, gc, json, glob, random, time, shutil, warnings, pickle
from collections import defaultdict
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
warnings.filterwarnings('ignore')
T_START = time.time()
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CUR_SIZE = [224]          # network input size of the model currently evaluated (ResNet18 = 96, the others 224)

def load_camelyon():
    from datasets import load_dataset, concatenate_datasets
    ds = load_dataset("wltjr1007/Camelyon17-WILDS"); parts = [ds[k] for k in ds.keys()]
    full = concatenate_datasets(parts) if len(parts) > 1 else parts[0]
    print(f"✓ Camelyon17-WILDS: {len(full):,} patches"); return full

def patient_split_by_centre(meta, val_frac, seed, centres=(0, 1, 2, 3, 4)):
    """Per-centre, patient-level split (identical convention to the Part-I probe scripts)."""
    tr, te = {}, {}
    for c in centres:
        pats = sorted(meta.loc[meta['center'] == c, 'patient'].unique().tolist())
        random.Random(seed + c).shuffle(pats)
        if len(pats) < 2: tr[c], te[c] = set(pats), set(); continue
        n_te = min(max(1, int(round(val_frac * len(pats)))), len(pats) - 1)
        te[c], tr[c] = set(pats[:n_te]), set(pats[n_te:])
    return tr, te

def stratified_sample(labels, n_per_class, seed):
    rng = np.random.RandomState(seed)
    pos, neg = np.where(labels == 1)[0], np.where(labels == 0)[0]
    rng.shuffle(pos); rng.shuffle(neg)
    k = min(n_per_class, len(pos), len(neg))
    if k < n_per_class: print(f"   ⚠ only {len(pos)} pos / {len(neg)} neg available -> {k}/class")
    idx = np.concatenate([pos[:k], neg[:k]]); rng.shuffle(idx); return idx

@torch.no_grad()
def embed(enc, X_u8, bs=128, amp=False, size=None):
    """uint8 [N,3,h,w] -> pooled features [N,D] (bilinear up-sampling to `size` on the GPU, ImageNet normalisation)."""
    out = []; size = size or CUR_SIZE[0]
    for i in range(0, len(X_u8), bs):
        x = X_u8[i:i + bs].to(DEVICE).float() / 255
        if x.shape[-1] != size: x = F.interpolate(x, size=(size, size), mode='bilinear', align_corners=False, antialias=True)
        x = (x - MEAN.to(DEVICE)) / STD.to(DEVICE)
        with torch.autocast('cuda', enabled=amp and DEVICE.type == 'cuda'):
            out.append(enc.encode(x).float().cpu())
    return torch.cat(out).numpy()

def load_uint8(ds, indices, size, desc):
    """Decode `indices` of an HF dataset to a uint8 tensor [n,3,size,size] (PIL bilinear resize if needed)."""
    from tqdm.auto import tqdm
    from torchvision import transforms
    tf = transforms.Resize((size, size))
    out = np.zeros((len(indices), size, size, 3), np.uint8)
    for k, i in enumerate(tqdm(indices, desc=desc)):
        im = ds[int(i)]['image']
        if im.mode != 'RGB': im = im.convert('RGB')
        if im.size != (size, size): im = tf(im)
        out[k] = np.asarray(im)
    return torch.from_numpy(out).permute(0, 3, 1, 2).contiguous()


# ---- frozen encoders / checkpoint loading (same code as the D7/D8 evaluations) ----
import os, gc, json, glob, random, time, shutil, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score
from tqdm import tqdm

warnings.filterwarnings('ignore')


class EC:
    RUN_FAMILIES = ['kaiko', 'lunit', 'phikon']
    SEEDS = [42, 43, 44]
    BUDGET_H = 11.0                       # stop starting new chunks when the next one would not fit
    HF_DATASET = "wltjr1007/Camelyon17-WILDS"
    CKPT_SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working/checkpoints"]
    KAGGLE_CKPT_DATASETS = ["sethu2954/adjc-kaiko-checkpoints", "sethu2954/adjc-lunit-checkpoints", "sethu2954/adjc-phikon-checkpoints"]   # fallback if not attached
    PRIOR_RESULT_ROOTS = ["/kaggle/input"]                                  # earlier runs' outputs (p2_eval_all/*.json) are reused
    FEATURE_DIR = "/kaggle/working/unused_features"
    RESULTS_DIR = "/kaggle/working/unused_results"
    KEEP_FEATURES = False

    EXTRACT_BATCH = 128
    NUM_WORKERS = 4
    ALL_CENTERS = [0, 1, 2, 3, 4]
    TEST_CENTER = 2
    LR_C, LR_MAX_ITER, PROBE_SEED = 1.0, 2000, 42
    BOOT_N, BOOT_SEED = 1000, 7
    LOCO_FOLDS = [0, 1, 2, 3, 4]
    CID_TRAIN_PER_CENTER, CID_TEST_PER_CENTER, CID_VAL_FRAC = 3000, 1000, 0.25

    FAMILIES = {   # kind, hub id / timm name, input size, fp16 extraction, Part-I LOCO of the original (pipeline check; None = ImageNet control)
        'resnet18':  dict(kind='resnet', hub=None, size=96,  amp=False, ref=0.9528, label='ImageNet ResNet18'),
        'vit_small': dict(kind='timm', hub='vit_small_patch16_224', size=224, amp=False, ref=None, label='ImageNet ViT-S/16'),
        'swin_tiny': dict(kind='timm', hub='swin_tiny_patch4_window7_224', size=224, amp=False, ref=None, label='ImageNet Swin-T'),
        'kaiko':     dict(kind='timm', hub='hf-hub:1aurent/vit_small_patch16_224.kaiko_ai_towards_large_pathology_fms', size=224, amp=False, ref=0.9873, label='Kaiko ViT-S/16'),
        'lunit':     dict(kind='timm', hub='hf-hub:1aurent/vit_small_patch16_224.lunit_dino', size=224, amp=False, ref=0.9814, label='Lunit ViT-S/16'),
        'phikon':    dict(kind='hf', hub='owkin/phikon', size=224, amp=True, ref=0.9818, label='Phikon ViT-B/16'),
    }
    ARMS = {}
    DEFAULT_ARMS = ['neighbor', 'self']
    BACKBONE = {'kaiko': 'vit_small', 'lunit': 'vit_small', 'phikon': 'phikon'}      # name used in the checkpoint file names
    PART1_OTHERS = "Part-I LOCO references: Stain Aug 0.9884 | Kaiko 0.9873 | Phikon 0.9818 | Lunit 0.9814 | ImageNet ResNet18 0.9528"

    @staticmethod
    def arms(fam):
        return EC.ARMS.get(fam, EC.DEFAULT_ARMS)

    @staticmethod
    def chunks(fam):
        """[(chunk_models)] : chunk 1 = original + seed 42, then one chunk per further seed.  Model name = '<family>__ctrl' / '<family>__<arm>_s<seed>'."""
        out = []
        for k, s in enumerate(EC.SEEDS):
            c = [f'{fam}__ctrl'] if k == 0 else []
            c += [f'{fam}__{a}_s{s}' for a in EC.arms(fam)]
            out.append(c)
        return out


def parse(name):
    fam, rest = name.split('__')
    if rest == 'ctrl':
        return fam, 'ctrl', None
    arm, s = rest.rsplit('_s', 1)
    return fam, arm, int(s)


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


# ---------------------------------------------------------------- encoders (same construction as Part I / the training notebooks)
class Encoder(nn.Module):
    def __init__(self, fam):
        super().__init__()
        spec = EC.FAMILIES[fam]; self.pool = None
        if spec['kind'] == 'resnet':
            net = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
            self.backbone = nn.Sequential(*list(net.children())[:-1]); self.pool = nn.AdaptiveAvgPool2d(1); self.dim = 512
        elif spec['kind'] == 'hf':
            from transformers import AutoModel
            class PhikonBackbone(nn.Module):
                def __init__(self):
                    super().__init__()
                    self.vit = AutoModel.from_pretrained(spec['hub'])
                def forward(self, x):
                    return self.vit(pixel_values=x).last_hidden_state[:, 0]           # CLS token, as in Part I
            self.backbone = PhikonBackbone(); self.dim = self.backbone.vit.config.hidden_size
        else:
            import timm
            self.backbone = timm.create_model(spec['hub'], pretrained=True, num_classes=0)
            self.dim = self.backbone.num_features

    @torch.no_grad()
    def encode(self, x):
        f = self.backbone(x)
        if f.dim() == 4:
            f = torch.flatten(self.pool(f), 1)
        elif f.dim() == 3:
            f = f.mean(1)
        return f


def ensure_checkpoint_roots():
    if glob.glob("/kaggle/input/**/adjclv4-*_best.pt", recursive=True):
        return
    import kagglehub
    for slug in EC.KAGGLE_CKPT_DATASETS:
        try:
            path = kagglehub.dataset_download(slug); print("✓ checkpoints downloaded via kagglehub:", path)
            EC.CKPT_SEARCH_ROOTS.append(path)
        except Exception as e:
            print(f"⚠ could not download {slug}: {e}")


def find_checkpoint(fam, arm, seed):
    fname = f"adjclv4-{fam}_{EC.BACKBONE[fam]}_{arm}_s{seed}_best.pt"
    for root in EC.CKPT_SEARCH_ROOTS:
        hits = glob.glob(os.path.join(root, '**', fname), recursive=True)
        if hits:
            return hits[0]
    return None


def build_model(name):
    fam, arm, seed = parse(name)
    if arm == 'ctrl':
        return Encoder(fam)
    path = find_checkpoint(fam, arm, seed)
    if path is None:
        return None
    enc = Encoder(fam)
    ck = torch.load(path, map_location='cpu')
    missing, unexpected = enc.load_state_dict(ck['model_state_dict'], strict=False)
    bad = [k for k in unexpected if not k.startswith(('head.', 'adv_head.'))]
    assert not missing and not bad, f"{name}: missing={missing[:3]} unexpected={bad[:3]}"
    print(f"  [{name}] loaded {os.path.basename(path)}  (epoch {ck.get('epoch','?')}, final loss {ck.get('best_val', float('nan')):.4f})")
    return enc


# ---------------------------------------------------------------- one-pass feature extraction
class MultiSizeDataset(Dataset):
    def __init__(self, hf, sizes):
        self.hf, self.sizes = hf, sorted(sizes)
        norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        self.tf = {s: transforms.Compose([transforms.Resize((s, s)), transforms.ToTensor(), norm]) for s in self.sizes}

    def __len__(self):
        return len(self.hf)

    def __getitem__(self, i):
        img = self.hf[int(i)]['image']
        if img.mode != 'RGB':
            img = img.convert('RGB')
        return {s: self.tf[s](img) for s in self.sizes}


def feat_path(name):
    return f"{EC.FEATURE_DIR}/{name}.npy"


def extract_missing(hf, names, device):
    todo = [n for n in names if not os.path.exists(feat_path(n))]
    if not todo:
        return
    mods = {}
    for n in todo:
        m = build_model(n)
        if m is None:
            print(f"  [{n}] checkpoint not found -> skipped")
            continue
        mods[n] = m.to(device).eval()
    if not mods:
        return
    fam = parse(todo[0])[0]; size = EC.FAMILIES[fam]['size']; amp = EC.FAMILIES[fam]['amp'] and device.type == 'cuda'
    N = len(hf)
    out = {n: np.empty((N, mods[n].dim), dtype=np.float32) for n in mods}
    loader = DataLoader(MultiSizeDataset(hf, {size}), batch_size=EC.EXTRACT_BATCH, shuffle=False,
                        num_workers=EC.NUM_WORKERS, pin_memory=device.type == 'cuda')
    pos, t0 = 0, time.time()
    for batch in tqdm(loader, desc=f"extract {fam}: {len(mods)} models"):
        x = batch[size].to(device, non_blocking=True); B = x.size(0)
        for n, m in mods.items():
            with torch.autocast('cuda', enabled=amp):
                f = m.encode(x)
            out[n][pos:pos + B] = f.float().cpu().numpy()
        pos += B
    assert pos == N, (pos, N)
    os.makedirs(EC.FEATURE_DIR, exist_ok=True)
    for n in mods:
        np.save(feat_path(n), out[n])
    print(f"✓ extracted {list(mods)} in {(time.time()-t0)/60:.1f} min")
    del mods, out; gc.collect(); torch.cuda.empty_cache()


# ---------------------------------------------------------------- evaluation (same recipe as Part I)
def bootstrap_auroc_by_patient(y, p, pat, n_boot, seed):
    pats = np.unique(pat)
    idx_by = {q: np.where(pat == q)[0] for q in pats}
    rng, vals = np.random.RandomState(seed), []
    for _ in range(n_boot):
        s = rng.choice(pats, size=len(pats), replace=True)
        idx = np.concatenate([idx_by[q] for q in s])
        if len(np.unique(y[idx])) < 2:
            continue
        vals.append(roc_auc_score(y[idx], p[idx]))
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return float(lo), float(hi)


def probe_auroc(X, y, centers, patients, train_c, test_c, with_ci=False):
    tr, te = np.isin(centers, train_c), centers == test_c
    sc = StandardScaler().fit(X[tr])
    clf = LogisticRegression(max_iter=EC.LR_MAX_ITER, C=EC.LR_C, class_weight='balanced',
                             random_state=EC.PROBE_SEED, n_jobs=-1)
    clf.fit(sc.transform(X[tr]), y[tr])
    prob = clf.predict_proba(sc.transform(X[te]))[:, 1]
    auc = roc_auc_score(y[te], prob)
    bal = balanced_accuracy_score(y[te], prob > 0.5)
    if with_ci:
        lo, hi = bootstrap_auroc_by_patient(y[te], prob, patients[te], EC.BOOT_N, EC.BOOT_SEED)
        return auc, bal, lo, hi
    return auc, bal


def center_id_probe(X, centers, patients):
    """Patch-level 5-way center-identity probe, patient-disjoint train/test within each center."""
    rng = np.random.RandomState(EC.PROBE_SEED)
    tr_idx, te_idx = [], []
    for c in EC.ALL_CENTERS:
        pats = sorted(np.unique(patients[centers == c]).tolist())
        random.Random(EC.PROBE_SEED + c).shuffle(pats)
        n_te = min(max(1, int(round(EC.CID_VAL_FRAC * len(pats)))), len(pats) - 1)
        te_p, tr_p = set(pats[:n_te]), set(pats[n_te:])
        for pool, n_want, store in ((np.where((centers == c) & np.isin(patients, list(tr_p)))[0], EC.CID_TRAIN_PER_CENTER, tr_idx),
                                    (np.where((centers == c) & np.isin(patients, list(te_p)))[0], EC.CID_TEST_PER_CENTER, te_idx)):
            store.append(rng.choice(pool, size=min(n_want, len(pool)), replace=False))
    tr_idx, te_idx = np.concatenate(tr_idx), np.concatenate(te_idx)
    sc = StandardScaler().fit(X[tr_idx])
    clf = LogisticRegression(max_iter=EC.LR_MAX_ITER, n_jobs=-1).fit(sc.transform(X[tr_idx]), centers[tr_idx])
    return 100 * accuracy_score(centers[te_idx], clf.predict(sc.transform(X[te_idx])))




# ====================================================================
# MODELS: original + {SimCLR control, self, adjacency, CCSM ms / fms, combined}  x seeds  (Kaiko / Lunit / Phikon)
# ====================================================================
FAM_ARMS = {'resnet18':  ['control', 'self', 'neighbor', 'ms', 'msrand', 'fms'],
            'vit_small': ['control', 'self', 'neighbor', 'ms', 'fms'],
            'swin_tiny': ['control', 'self', 'neighbor', 'ms', 'fms'],
            'kaiko':  ['control', 'self', 'neighbor', 'ms', 'fms'],
            'lunit':  ['control', 'self', 'neighbor', 'ms', 'fms', 'comb'],
            'phikon': ['control', 'self', 'neighbor', 'ms', 'fms']}
ARM_LABEL = {'ctrl': 'ORIGINAL pretrained weights', 'msrand': 'CCSM: MixStyle, random partners', 'control': 'SimCLR control (CCSM recipe)', 'self': 'self control (adjacency recipe)', 'neighbor': 'ADJACENCY (Method 1)',
             'ms': 'CCSM: MixStyle', 'fms': 'CCSM: MixStyle + Fourier', 'comb': 'COMBINED adjacency + CCSM'}
ARM_ORDER = ['ctrl', 'control', 'self', 'neighbor', 'ms', 'msrand', 'fms', 'comb']
ADJ_SLUGS = {'resnet18': "peesarisathvikreddy/adjclv2-checkpoints", 'vit_small': "peesarisathvikreddy/adjclv2-checkpoints", 'swin_tiny': "peesarisathvikreddy/adjclv2-checkpoints",
             'kaiko': "sethu2954/adjc-kaiko-checkpoints", 'lunit': "sethu2954/adjc-lunit-checkpoints", 'phikon': "sethu2954/adjc-phikon-checkpoints"}
COMB_SLUG = "badrinathachanta/ablation-combined-checkpoints"      # Lunit combined (adjacency + CCSM) checkpoints from ABL5
CCSM_SLUGS = {'resnet18': "badrinathachanta/ccsm-checkpoints-1", 'vit_small': "badrinathachanta/ccsm-checkpoints-1", 'swin_tiny': "badrinathachanta/ccsm-checkpoints-1",
              'kaiko': "badrinathachanta/ccsm-checkpoints-2", 'lunit': "badrinathachanta/ccsm-checkpoints-2", 'phikon': "badrinathachanta/ccsm-checkpoints-2"}

def ckpt_fname(fam, arm, seed):
    if arm in ('self', 'neighbor') and fam in ('resnet18', 'vit_small', 'swin_tiny'): return f"adjclv2_{fam}_{arm}_s{seed}_best.pt"      # adjacency v2 recipe (ResNet18 / ViT-S / Swin-T)
    if arm in ('self', 'neighbor'): return f"adjclv4-{fam}_{'phikon' if fam == 'phikon' else 'vit_small'}_{arm}_s{seed}_best.pt"
    if arm in ('control', 'ms', 'msrand', 'fms'): return f"ccsm_{fam}_{arm}_s{seed}_best.pt"
    if arm == 'comb': return f"abl5-comb_vit_small_neighbor_s{seed}_best.pt"
    raise ValueError(arm)

def ensure_checkpoint_roots():
    import kagglehub
    for fam in RUN_FAMILIES:
        pairs = [('ccsm', CCSM_SLUGS[fam])] + ([('adjacency', ADJ_SLUGS[fam])] if fam in ADJ_SLUGS else []) + ([('comb', COMB_SLUG)] if fam == 'lunit' else [])
        for kind, slug in pairs:
            probe = ckpt_fname(fam, {'adjacency': 'neighbor', 'comb': 'comb'}.get(kind, 'ms'), SEEDS[0])
            if glob.glob(os.path.join('/kaggle/input', '**', probe), recursive=True): continue
            try:
                path = kagglehub.dataset_download(slug); print("✓ checkpoints downloaded via kagglehub:", slug); EC.CKPT_SEARCH_ROOTS.append(path)
            except Exception as e:
                print(f"⚠ could not download {slug}: {e}")

def find_checkpoint(fam, arm, seed):
    fname = ckpt_fname(fam, arm, seed)
    for root in ['/kaggle/input'] + list(EC.CKPT_SEARCH_ROOTS):
        hits = glob.glob(os.path.join(root, '**', fname), recursive=True)
        if hits: return hits[0]
    return None

def model_names():
    out = []
    for fam in RUN_FAMILIES:
        out.append(f'{fam}__ctrl')
        for s in SEEDS: out += [f'{fam}__{a}_s{s}' for a in FAM_ARMS[fam]]
    return out


def have_result(name): return os.path.exists(f"{RES_DIR}/{name}.json")
def prior_results():
    n = 0
    for p in glob.glob(f'/kaggle/input/**/{os.path.basename(RES_DIR)}/*.json', recursive=True):
        dst = os.path.join(RES_DIR, os.path.basename(p))
        if not os.path.exists(dst): shutil.copy2(p, dst); n += 1
    if n: print(f"✓ reused {n} finished results from an earlier run")

def _ms(v):
    v = np.array(v, dtype=float); return v.mean(), (v.std(ddof=1) if len(v) > 1 else float('nan'))

def run_all(process_fn, device):
    """process_fn(name, encoder) -> dict (JSON-able).  Resumable; stops cleanly before BUDGET_H."""
    T0 = time.time(); last = 0.0
    os.makedirs(RES_DIR, exist_ok=True); prior_results(); ensure_checkpoint_roots()
    names = model_names(); todo = [n for n in names if not have_result(n)]
    print(f"{len(names) - len(todo)}/{len(names)} models already done")
    for n in todo:
        el = (time.time() - T0) / 3600
        if el + max(last, 0.15) > BUDGET_H:
            print(f"\n⏸ time budget reached ({el:.1f} h). Save this version and re-run with this OUTPUT attached as an input to finish the rest."); break
        t0 = time.time()
        enc = build_model(n)
        if enc is None:
            print(f"  [{n}] checkpoint not found -> skipped"); continue
        enc = enc.to(device).eval(); CUR_SIZE[0] = EC.FAMILIES[n.split('__')[0]]['size']
        res = process_fn(n, enc); res['model'] = n
        json.dump(res, open(f"{RES_DIR}/{n}.json", 'w'))
        del enc; gc.collect(); torch.cuda.empty_cache()
        last = (time.time() - t0) / 3600


"""
Protocol follows the Part-I probe battery: 1500 clean 3x3 windows per centre (70% from train patients / 30% from held-out patients, per centre, seed 42), probes 2/4/5 on the frozen
patch features (a patch-level model encodes each of the 9 patches independently, the window feature is their mean).  Deviation from Part-I: features are standardised (StandardScaler)
and logistic regression uses max_iter=500 so that all models converge in comparable time; MLP probe: one hidden layer of 128 units.  Patient probe: random 70/30 split of the
patches (as in Part-I; patches of one window can fall on both sides).
"""
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from torchvision import transforms
from tqdm.auto import tqdm

RUN_FAMILIES = ['phikon']
SEEDS = [42, 43, 44]
BUDGET_H = 11.0
RES_DIR = "/kaggle/working/abl_probes_c"
CACHE_DIR = "/kaggle/working/cache"
SEED = 42; GRID = 3; WIN_PER_CENTRE = 1500; VAL_FRAC = 0.3
os.makedirs(CACHE_DIR, exist_ok=True)

# ---- spatial index (copied from the Part-I probe script) ----
class Lattice:
    __slots__ = ('patient', 'node', 'center', 'idx_grid', 'valid_centers')

    def __init__(self, patient, node, center, patches, grid_size):
        self.patient, self.node, self.center = patient, node, center
        xs = sorted({p['x'] for p in patches})
        ys = sorted({p['y'] for p in patches})
        x_to_c = {x: i for i, x in enumerate(xs)}
        y_to_r = {y: i for i, y in enumerate(ys)}
        g = np.full((len(ys), len(xs)), -1, dtype=np.int64)
        for p in patches:
            g[y_to_r[p['y']], x_to_c[p['x']]] = p['idx']
        self.idx_grid = g

        H, W = g.shape
        valid = []
        if H >= grid_size and W >= grid_size:
            filled = g >= 0
            integral = np.zeros((H + 1, W + 1), dtype=np.int64)
            integral[1:, 1:] = filled.astype(np.int64).cumsum(0).cumsum(1)
            for r in range(0, H - grid_size + 1):
                for c in range(0, W - grid_size + 1):
                    s = (integral[r + grid_size, c + grid_size]
                         - integral[r, c + grid_size]
                         - integral[r + grid_size, c]
                         + integral[r, c])
                    if s == grid_size * grid_size:
                        valid.append((r, c))
        self.valid_centers = np.array(valid, dtype=np.int32) if valid else np.zeros((0, 2), dtype=np.int32)

    def window(self, r, c, grid_size):
        return self.idx_grid[r:r + grid_size, c:c + grid_size].ravel()


class SpatialIndex:
    def __init__(self, dataset, grid_size, cache_path=None):
        self.grid_size = grid_size
        if cache_path and os.path.exists(cache_path):
            with open(cache_path, 'rb') as f:
                self.lattices = pickle.load(f)
            print(f"OK loaded {len(self.lattices)} lattices from cache")
        else:
            self.lattices = self._build(dataset)
            if cache_path:
                with open(cache_path, 'wb') as f:
                    pickle.dump(self.lattices, f)
                print(f"OK cached -> {cache_path}")

    def _build(self, dataset):
        print("Building lattices...")
        meta = dataset.remove_columns(['image']).to_pandas()
        groups = defaultdict(list)
        for row in tqdm(meta.itertuples(index=True), total=len(meta), desc="index"):
            groups[(int(row.patient), int(row.node), int(row.center))].append(
                {'x': int(row.x_coord), 'y': int(row.y_coord), 'idx': int(row.Index)})
        lattices = []
        for (pat, node, ctr), patches in tqdm(groups.items(), desc="lattices"):
            lat = Lattice(pat, node, ctr, patches, self.grid_size)
            if len(lat.valid_centers):
                lattices.append(lat)
        return lattices

    def subset(self, centers, patients=None):
        pats = set(patients) if patients is not None else None
        return [lat for lat in self.lattices
                if lat.center in centers and (pats is None or lat.patient in pats)]




def build_windows(index):
    n_tr = int(round(WIN_PER_CENTRE * (1 - VAL_FRAC))); n_te = WIN_PER_CENTRE - n_tr
    recs = []                                                   # (window patch indices[9], centre, patient, is_test)
    for c in range(5):
        lats = index.subset([c])
        if not lats: continue
        pats = sorted({l.patient for l in lats}); random.Random(SEED + c).shuffle(pats)
        n_tp = min(max(1, int(round(VAL_FRAC * len(pats)))), len(pats) - 1) if len(pats) >= 2 else 0
        te_p, tr_p = set(pats[:n_tp]), set(pats[n_tp:]) if n_tp else set(pats)
        for lat_set, n, seed, is_test in ((index.subset([c], tr_p), n_tr, SEED + c, 0), (index.subset([c], te_p) if te_p else [], n_te, SEED + 1000 + c, 1)):
            if not lat_set: continue
            rng = np.random.RandomState(seed); counts = np.array([len(l.valid_centers) for l in lat_set], float); p = counts / counts.sum()
            for _ in range(n):
                li = int(rng.choice(len(lat_set), p=p)); lat = lat_set[li]; ci = int(rng.randint(len(lat.valid_centers)))
                r, cc = lat.valid_centers[ci]
                recs.append((lat.window(int(r), int(cc), GRID).copy(), c, lat.patient, is_test))
    return recs

DATA = {}
def prepare():
    cp = f"{CACHE_DIR}/probe_data.pt"
    for p in [cp] + glob.glob('/kaggle/input/**/probe_data.pt', recursive=True):
        if os.path.exists(p):
            print("✓ loading cached probe data", p); DATA.update(torch.load(p, weights_only=False)); return
    cam = load_camelyon()
    index = SpatialIndex(cam, GRID, cache_path=f"{CACHE_DIR}/probe_lattices_3x3.pkl")
    recs = build_windows(index)
    uniq = np.unique(np.concatenate([r[0] for r in recs])); row_of = {int(u): i for i, u in enumerate(uniq)}
    imgs = load_uint8(cam, uniq, 96, 'patches');
    tf = transforms.Resize((64, 64)); colour = np.zeros((len(uniq), 6), np.float32)
    for i in tqdm(range(len(uniq)), desc='colour statistics'):
        x = F.interpolate(imgs[i:i + 1].float() / 255, size=(64, 64), mode='bilinear', align_corners=False)[0]
        colour[i] = torch.cat([x.mean((1, 2)), x.std((1, 2))]).numpy()
    rows = np.array([[row_of[int(k)] for k in r[0]] for r in recs])            # [W, 9]
    DATA.update(imgs=imgs, colour=colour, rows=rows, centre=np.array([r[1] for r in recs]), patient=np.array([r[2] for r in recs]), is_test=np.array([r[3] for r in recs]))
    torch.save(dict(DATA), cp); print(f"✓ {len(recs)} windows, {len(uniq)} unique patches cached")

def lr_acc(Xtr, ytr, Xte, yte, max_iter=500):
    sc = StandardScaler().fit(Xtr)
    clf = LogisticRegression(max_iter=max_iter).fit(sc.transform(Xtr), ytr)
    return 100 * accuracy_score(yte, clf.predict(sc.transform(Xte)))

def colour_baseline():
    """Model-free hospital baseline from 6-d colour statistics (mean/std per RGB channel of a 64x64 resize): per patch (standardised) and per 3x3 window (mean of its 9 patches), standardised and raw."""
    d = DATA; W = d['rows'].shape[0]
    X = d['colour'][d['rows'].reshape(-1)]; c = np.repeat(d['centre'], 9); t = np.repeat(d['is_test'], 9).astype(bool)
    Xw = d['colour'][d['rows']].mean(1); cw = d['centre']; tw = d['is_test'].astype(bool)
    raw = 100 * accuracy_score(cw[tw], LogisticRegression(max_iter=500).fit(Xw[~tw], cw[~tw]).predict(Xw[tw]))
    return {'patch_standardised': lr_acc(X[~t], c[~t], X[t], c[t]), 'window_standardised': lr_acc(Xw[~tw], cw[~tw], Xw[tw], cw[tw]), 'window_raw_unscaled': raw}

def process(name, enc):
    fam = name.split('__')[0]; amp = EC.FAMILIES[fam]['amp']; t0 = time.time(); d = DATA
    Fu = embed(enc, d['imgs'], amp=amp)                                            # [U, D] features of all unique patches
    W = d['rows'].shape[0]; emb = Fu[d['rows'].reshape(-1)]                        # [W*9, D]
    win = emb.reshape(W, 9, -1).mean(1)
    t = d['is_test'].astype(bool); tp = np.repeat(t, 9)
    centre_w, centre_p = d['centre'], np.repeat(d['centre'], 9)
    pos = np.tile(np.arange(9), W); pat_p = np.repeat(d['patient'], 9)
    res = {}
    sc = StandardScaler().fit(win[~t]); a, b = sc.transform(win[~t]), sc.transform(win[t])
    res['centre_logreg'] = 100 * accuracy_score(centre_w[t], LogisticRegression(max_iter=500).fit(a, centre_w[~t]).predict(b))
    res['centre_mlp'] = 100 * accuracy_score(centre_w[t], MLPClassifier(hidden_layer_sizes=(128,), max_iter=500, random_state=SEED).fit(a, centre_w[~t]).predict(b))
    res['position'] = lr_acc(emb[~tp], pos[~tp], emb[tp], pos[tp])
    rng = np.random.RandomState(SEED); perm = rng.permutation(len(emb)); cut = int(0.7 * len(emb)); tr, te = perm[:cut], perm[cut:]
    res['patient'] = lr_acc(emb[tr], pat_p[tr], emb[te], pat_p[te])
    res['n_patients'] = int(len(np.unique(pat_p)))
    print(f"  [{name}] centre(logreg) {res['centre_logreg']:.1f}% | centre(MLP) {res['centre_mlp']:.1f}% | position {res['position']:.1f}% | patient {res['patient']:.1f}% | {(time.time()-t0)/60:.1f} min")
    return res

def summarize():
    res = {}
    for p in glob.glob(f"{RES_DIR}/*.json"):
        if os.path.basename(p).startswith('colour_baseline'): continue
        r = json.load(open(p)); res[r['model']] = r
    cb = json.load(open(f"{RES_DIR}/colour_baselines.json")) if os.path.exists(f"{RES_DIR}/colour_baselines.json") else {}
    print("\n" + "=" * 118 + "\nPROBE BATTERY on frozen features  (accuracy %; mean ± std over seeds).  Chance: centre 20 | position 11.1 | patient ~1/#patients\n" + "=" * 118)
    print("  raw colour statistics -> centre (no model): " + " | ".join(f"{k} {v:.1f}%" for k, v in cb.items()) + "   (Part-I reported 87.5%; its exact feature scaling is not recorded - compare the window-level numbers)")
    rows = []
    for fam in RUN_FAMILIES:
        print(f"\n--- {fam} ---\n  {'':34s}{'centre LR':>14s}{'centre MLP':>14s}{'position':>14s}{'patient':>14s}")
        per = {a: {} for a in ARM_ORDER}
        for n, r in res.items():
            f, a, s = parse(n)
            if f == fam: per[a][s] = r
        for a in ARM_ORDER:
            dd = per[a]
            if not dd: continue
            vals = list(dd.values()); line = f"  {ARM_LABEL[a]:34s}"; row = dict(family=fam, method=ARM_LABEL[a], seeds=len(vals))
            for k in ('centre_logreg', 'centre_mlp', 'position', 'patient'):
                m, sd = _ms([v[k] for v in vals]); c = f"{m:6.1f}" + (f"±{sd:4.1f}" if len(vals) > 1 else "     "); line += f"{c:>14s}"; row[k] = c.strip()
            print(line); rows.append(row)
        for a, b, lab in [('neighbor', 'self', 'adjacency minus self'), ('neighbor', 'ctrl', 'adjacency minus ORIGINAL'), ('fms', 'control', 'CCSM(MS+F) minus SimCLR control'),
                          ('fms', 'ctrl', 'CCSM(MS+F) minus ORIGINAL'), ('comb', 'neighbor', 'combined minus adjacency')]:
            if per.get(a) and per.get(b):
                rb = (lambda s_, b=b: per[b][None]) if b == 'ctrl' else (lambda s_, b=b: per[b][s_])
                common = sorted(per[a]) if b == 'ctrl' else sorted(set(per[a]) & set(per[b])); out = []
                for k in ('centre_logreg', 'centre_mlp', 'position', 'patient'):
                    g = [per[a][s][k] - rb(s)[k] for s in common]; out.append(f"{k} {np.mean(g):+5.1f} ({sum(x < 0 for x in g)}/{len(g)} lower)")
                print(f"      {lab:34s} " + " | ".join(out))
    pd.DataFrame(rows).to_csv(f"{RES_DIR}/probe_summary.csv", index=False)
    print("\nHigher centre accuracy = more hospital-identifiable features (Part-I: raw colour alone 87.5%).  Position / patient accuracy show how much layout / identity the features retain.")

if True:
    os.makedirs(RES_DIR, exist_ok=True)
    prepare()
    if not os.path.exists(f"{RES_DIR}/colour_baselines.json"):
        json.dump(colour_baseline(), open(f"{RES_DIR}/colour_baselines.json", 'w'))
    run_all(process, DEVICE)
    summarize()

README.md:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

data/train-00000-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  470MB            

data/train-00000-of-00014.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  469MB            

data/train-00001-of-00014.parquet: downloading bytes:           |  0.00B            

...

  [phikon__neighbor_s44] loaded adjclv4-phikon_phikon_neighbor_s44_best.pt  (epoch 24, final loss 0.6028)
  [phikon__neighbor_s44] centre(logreg) 87.2% | centre(MLP) 77.5% | position 11.7% | patient 99.9% | 3.3 min


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

  [phikon__ms_s44] loaded ccsm_phikon_ms_s44_best.pt  (epoch 6, final loss 0.0467)
  [phikon__ms_s44] centre(logreg) 97.3% | centre(MLP) 97.4% | position 11.1% | patient 100.0% | 3.2 min


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

  [phikon__fms_s44] loaded ccsm_phikon_fms_s44_best.pt  (epoch 6, final loss 0.0713)
  [phikon__fms_s44] centre(logreg) 89.3% | centre(MLP) 87.6% | position 11.4% | patient 100.0% | 3.3 min

PROBE BATTERY on frozen features  (accuracy %; mean ± std over seeds).  Chance: centre 20 | position 11.1 | patient ~1/#patients
  raw colour statistics -> centre (no model): patch_standardised 97.7% | window_standardised 96.1% | window_raw_unscaled 73.8%   (Part-I reported 87.5%; its exact feature scaling is not recorded - compare the window-level numbers)

--- phikon ---
                                         centre LR    centre MLP      position       patient
  ORIGINAL pretrained weights            98.7          99.2          11.7         100.0     
  SimCLR control (CCSM recipe)           97.8± 0.5     97.2± 0.6     11.0± 0.0    100.0± 0.0
  self control (adjacency recipe)        85.1± 0.6     77.7± 0.7     11.7± 0.2     99.7± 0.0
  ADJACENCY (Method 1)                   86.1± 1.1     79.2± 